# Chess Engine with TensorFlow

## Dataset

In [1]:
import os
# inspired by Github user Skripkon 
# https://github.com/Skripkon/chess-engine/blob/main/engines/tensorflow/train_and_predict.ipynb
# used to train models from scratch
# took around 3 hours for 20000 games

# get the game files
files = [file for file in os.listdir("new_data_2500_Elo") if file.endswith(".pgn")]

In [2]:
from chess import pgn

# load the games
def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
            
    return games

In [3]:
from tqdm import tqdm
# write all the games together 

games = []
for file in tqdm(files):
    games.extend(load_pgn(f"new_data_2500_Elo/{file}"))

100%|██████████| 15/15 [00:42<00:00,  2.85s/it]


In [4]:
len(games) # check how many 

14237

## Build & train a neural network

In [5]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from chess import Board
import tensorflow as tf

In [6]:
#translating the board into a matrix for the predition process 
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix

# create the inputs based off game position and next move 
def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y

# encode all the moves
def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

In [7]:
# create the training data
X, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = np.array(y)
X = np.array(X)

In [8]:
print(len(move_to_int))

1964


In [ ]:

# train the model
model = tf.keras.models.Sequential([
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu', input_shape=(8, 8, 12)),
    tf.keras.layers.Conv2D(128, (3, 3), activation='relu'),
    #tf.keras.layers.Conv2D(256, (3, 3), activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(2048, activation='relu', name='new_dense1'),
    tf.keras.layers.Dropout(0.4),
    tf.keras.layers.Dense(len(move_to_int), activation='softmax', name='new_output')
])
model.compile(optimizer = tf.keras.optimizers.Adam(learning_rate=0.0001), loss='sparse_categorical_crossentropy', metrics=['accuracy'])
model.summary()

callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_accuracy',
        mode='max',
        patience=7,
        restore_best_weights=True,
        verbose=1
    )
]

model.fit(X, y, epochs=50, validation_split=0.1, batch_size=128, callbacks=callbacks)
# save the model
model.save("HighEloBaller/SSMF_50EPOCHS.keras")

import pickle
# save the encoding
with open("HighEloBaller/move_to_int.pkl", "wb") as f:
    pickle.dump(move_to_int, f)
int_to_move = {v: k for k, v in move_to_int.items()}
with open("HighEloBaller/int_to_move.pkl", "wb") as f:
    pickle.dump(int_to_move, f)
# configuration 
config = {
    "epochs": 50,
    "batch_size": 128,
    "validation_split": 0.1,
    "optimizer": "Adam",
    "input_shape": (8, 8, 12),
}
# save the configurations
with open("HighEloBaller/train_config.json", "w") as f:
    import json
    json.dump(config, f, indent=4)



Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_dense1 (Dense)              │ (None, 2048)           │     4,196,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ new_output (Dense)              │ (None, 1964)           │     4,024,236 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 8,301,420 (31.67 MB)

 Trainable params: 8,301,420 (31.67 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 323s 23ms/step - accuracy: 0.0449 - loss: 5.3495 - val_accuracy: 0.0648 - val_loss: 4.4842
Epoch 2/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 318s 23ms/step - accuracy: 0.0654 - loss: 4.4170 - val_accuracy: 0.0775 - val_loss: 4.1761
Epoch 3/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 316s 22ms/step - accuracy: 0.0771 - loss: 4.1721 - val_accuracy: 0.0837 - val_loss: 4.0457
Epoch 4/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 314s 22ms/step - accuracy: 0.0873 - loss: 4.0278 - val_accuracy: 0.0885 - val_loss: 3.9780
Epoch 5/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 318s 23ms/step - accuracy: 0.0962 - loss: 3.9233 - val_accuracy: 0.0921 - val_loss: 3.9309
Epoch 6/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 320s 23ms/step - accuracy: 0.1042 - loss: 3.8404 - val_accuracy: 0.0950 - val_loss: 3.8924
Epoch 7/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 325s 23ms/step - accuracy: 0.1114 - loss: 3.7705 - val_accuracy: 0.0976 - val_loss: 3.8737
Epoch 8/50
14069/14069 ━━━━━━━━━━━━━━━━━━━━ 324s 23ms/s